## 4. Domain-Aware Missing Value Imputation
Applying industry-standard domain strategies:
1. **Structural Zeros**: Structural absences (e.g., unticketed monuments have fee = 0; regional airfields have international passengers = 0).
2. **Rounded Group-Conditional Medians (`fill_int_median`)**: Integer columns (`Int64`) filled with stratified group medians rounded to whole numbers to prevent dtype errors like `Invalid value '63.5' for dtype 'Int64'`.
3. **Explicit Category Tokens**: Preserving unobserved information (`'Unclassified'`, `'Unknown'`) instead of biasing frequencies toward the statistical mode.

*Each dataset is imputed in a separate cell with `.head()` and null verification.*


In [ ]:
import pandas as pd
import numpy as np

# Load datasets and prepare types for imputation
df_district_footfall = pd.read_csv('../data/raw/01_district_wise_tourist_footfall_2022_2024.csv')
df_nidhi_hospitality = pd.read_csv('../data/raw/02_nidhi_hospitality_establishments_registry.csv')
df_asi_monuments = pd.read_csv('../data/raw/03_asi_protected_monuments_inventory.csv')
df_inbound_survey = pd.read_csv('../data/raw/04_inbound_tourist_survey_microdata_2023_2024.csv')
df_airport_traffic = pd.read_csv('../data/raw/05_airport_passenger_traffic_monthly_2022_2024.csv')
df_macro_overview = pd.read_csv('../data/raw/06_annual_macro_overview_2022_2024.csv')
df_fta_summary = pd.read_csv('../data/raw/07_country_wise_fta_summary_2022_2024.csv')
df_state_ut_summary = pd.read_csv('../data/raw/08_state_ut_consolidated_summary_2022_2023.csv')

# Initial casts required for categorical accessors
df_district_footfall['Tourism_Category'] = df_district_footfall['Tourism_Category'].astype('category')
df_district_footfall['State_UT'] = df_district_footfall['State_UT'].astype('category')
df_nidhi_hospitality['Star_Classification'] = df_nidhi_hospitality['Star_Classification'].astype('category')
df_nidhi_hospitality['Eco_Green_Certified'] = df_nidhi_hospitality['Eco_Green_Certified'].astype('category')
df_nidhi_hospitality['Pincode'] = df_nidhi_hospitality['Pincode'].astype('Int64').astype('string')
df_asi_monuments['Location_Setting'] = df_asi_monuments['Location_Setting'].astype('category')
df_asi_monuments['Era_Period'] = df_asi_monuments['Era_Period'].astype('category')
df_inbound_survey['Gender'] = df_inbound_survey['Gender'].astype('category')
df_inbound_survey['Would_Revisit_India'] = df_inbound_survey['Would_Revisit_India'].astype('category')
print('Datasets prepared for domain-aware missing value imputation.')


### 4.1 Impute Missing Values: `df_district_footfall`


In [116]:
# Helper function to round medians for Int64 columns
def fill_int_median(s):
    med = s.median()
    if pd.notna(med):
        return s.fillna(int(round(med)))
    return s

# 1. Foreign Visitors: Unreported/remote districts -> structural 0
df_district_footfall['Foreign_Visitors'] = df_district_footfall['Foreign_Visitors'].fillna(0)

# 2. Registered Hotels Count: Rounded median by State & Category
df_district_footfall['Registered_Hotels_Count'] = (
    df_district_footfall.groupby(['State_UT', 'Tourism_Category'])['Registered_Hotels_Count']
    .transform(fill_int_median)
    .fillna(int(round(df_district_footfall['Registered_Hotels_Count'].median())))
    .astype('Int64')
)

# 3. Average Stay Duration: Conditional float median by Tourism Category
df_district_footfall['Avg_Stay_Duration_Days'] = (
    df_district_footfall.groupby('Tourism_Category')['Avg_Stay_Duration_Days']
    .transform(lambda s: s.fillna(s.median()))
)

# 4. Tourist Complaints: 0 where unrecorded or no tourist desk exists
df_district_footfall['Reported_Tourist_Complaints'] = df_district_footfall['Reported_Tourist_Complaints'].fillna(0)

print('df_district_footfall missing values remaining:', df_district_footfall.isnull().sum().sum())
# Verify with .head()
display(df_district_footfall[['District', 'State_UT', 'Year', 'Foreign_Visitors', 'Registered_Hotels_Count', 'Avg_Stay_Duration_Days', 'Reported_Tourist_Complaints']].head(3))


df_district_footfall missing values remaining: 0


,District,State_UT,Year,Foreign_Visitors,Registered_Hotels_Count,Avg_Stay_Duration_Days,Reported_Tourist_Complaints
0,Agra,Uttar Pradesh,2022,48112,401,3.6,2
1,Agra,Uttar Pradesh,2023,54991,401,1.3,34
2,Agra,Uttar Pradesh,2024,52054,401,4.1,27


### 4.2 Impute Missing Values: `df_nidhi_hospitality`


In [117]:
# 1. Max Tariff: For fixed-rate homestays/rooms, Max Tariff == Min Tariff
df_nidhi_hospitality['Room_Tariff_Max_INR'] = (
    df_nidhi_hospitality['Room_Tariff_Max_INR'].fillna(df_nidhi_hospitality['Room_Tariff_Min_INR'])
)

# 2. Star Classification: Explicit 'Unclassified' category
if 'Unclassified' not in df_nidhi_hospitality['Star_Classification'].cat.categories:
    df_nidhi_hospitality['Star_Classification'] = df_nidhi_hospitality['Star_Classification'].cat.add_categories(['Unclassified'])
df_nidhi_hospitality['Star_Classification'] = df_nidhi_hospitality['Star_Classification'].fillna('Unclassified')

# 3. Eco Green Certified: Explicit 'No'
if 'No' not in df_nidhi_hospitality['Eco_Green_Certified'].cat.categories:
    df_nidhi_hospitality['Eco_Green_Certified'] = df_nidhi_hospitality['Eco_Green_Certified'].cat.add_categories(['No'])
df_nidhi_hospitality['Eco_Green_Certified'] = df_nidhi_hospitality['Eco_Green_Certified'].fillna('No')

# 4. Pincode: Hierarchical mode by District_City
district_pincode_mode = df_nidhi_hospitality.groupby('District_City')['Pincode'].transform(
    lambda s: s.mode().iloc[0] if not s.mode().empty else 'Unknown'
)
df_nidhi_hospitality['Pincode'] = df_nidhi_hospitality['Pincode'].fillna(district_pincode_mode).fillna('Unknown')

# 5. Audit Quality Score: Conditional median + Missing Indicator flag
df_nidhi_hospitality['Audit_Score_Was_Missing'] = df_nidhi_hospitality['Audit_Quality_Score'].isna()
df_nidhi_hospitality['Audit_Quality_Score'] = (
    df_nidhi_hospitality.groupby('Accommodation_Type')['Audit_Quality_Score']
    .transform(lambda s: s.fillna(s.median()))
    .fillna(df_nidhi_hospitality['Audit_Quality_Score'].median())
)

print('df_nidhi_hospitality missing values remaining:', df_nidhi_hospitality.isnull().sum().sum())
# Verify with .head()
display(df_nidhi_hospitality[['Establishment_Name', 'Star_Classification', 'Pincode', 'Room_Tariff_Min_INR', 'Room_Tariff_Max_INR', 'Audit_Quality_Score', 'Audit_Score_Was_Missing']].head(3))


df_nidhi_hospitality missing values remaining: 0


,Establishment_Name,Star_Classification,Pincode,Room_Tariff_Min_INR,Room_Tariff_Max_INR,Audit_Quality_Score,Audit_Score_Was_Missing
0,Heritage Chandigarh Suites,Unclassified,497236,2500,2500,78.4,True
1,Mountain Khurda Boutique Stay,Unclassified,543168,2500,6500,75.1,True
2,Mountain Bhagalpur Palace,3 Star,244980,7500,11500,79.4,True


### 4.3 Impute Missing Values: `df_asi_monuments`


In [118]:
# 1. Entry Fees: Structurally free for non-ticketed sites -> 0
df_asi_monuments['Entry_Fee_Domestic_INR'] = df_asi_monuments['Entry_Fee_Domestic_INR'].fillna(0)
df_asi_monuments['Entry_Fee_Foreign_INR'] = df_asi_monuments['Entry_Fee_Foreign_INR'].fillna(0)

# 2. Footfall: Rounded group median by Architectural Type
for footfall_col in ['Estimated_Annual_Footfall_2023', 'Estimated_Annual_Footfall_2025', 'Estimated_Annual_Footfall_2026']:
    df_asi_monuments[footfall_col] = (
        df_asi_monuments.groupby('Architectural_Type')[footfall_col]
        .transform(fill_int_median)
        .fillna(int(round(df_asi_monuments[footfall_col].median())))
        .astype('Int64')
    )

# 3. Category Fallbacks: Explicit 'Unknown' and 'Unspecified'
for col, fallback in [('Location_Setting', 'Unknown'), ('Era_Period', 'Unspecified')]:
    if fallback not in df_asi_monuments[col].cat.categories:
        df_asi_monuments[col] = df_asi_monuments[col].cat.add_categories([fallback])
    df_asi_monuments[col] = df_asi_monuments[col].fillna(fallback)

print('df_asi_monuments missing values remaining:', df_asi_monuments.isnull().sum().sum())
# Verify with .head()
display(df_asi_monuments[['Monument_Name', 'Architectural_Type', 'Location_Setting', 'Entry_Fee_Domestic_INR', 'Estimated_Annual_Footfall_2026']].head(3))


df_asi_monuments missing values remaining: 0


,Monument_Name,Architectural_Type,Location_Setting,Entry_Fee_Domestic_INR,Estimated_Annual_Footfall_2026
0,Taj Mahal,Stupa/Monastery,Urban,40,7738564
1,Qutub Minar,Temple/Shrine,Forest/Sanctuary,25,7621253
2,Red Fort,Fort/Bastion,Forest/Sanctuary,50,3592065


### 4.4 Impute Missing Values: `df_inbound_survey`


In [119]:
# 1. Spend: Group median by Region of Origin & Primary Trip Purpose
df_inbound_survey['Estimated_Total_Spend_USD'] = (
    df_inbound_survey.groupby(['Region_of_Origin', 'Primary_Trip_Purpose'])['Estimated_Total_Spend_USD']
    .transform(lambda s: s.fillna(s.median()))
    .fillna(df_inbound_survey['Estimated_Total_Spend_USD'].median())
)
df_inbound_survey['Daily_Spend_INR'] = (
    df_inbound_survey.groupby(['Region_of_Origin', 'Primary_Trip_Purpose'])['Daily_Spend_INR']
    .transform(lambda s: s.fillna(s.median()))
    .fillna(df_inbound_survey['Daily_Spend_INR'].median())
)

# 2. Rating Scales: Rounded integer medians
df_inbound_survey['Safety_Perception_Rating_1to5'] = (
    df_inbound_survey.groupby('Port_of_Exit')['Safety_Perception_Rating_1to5']
    .transform(fill_int_median)
    .fillna(int(round(df_inbound_survey['Safety_Perception_Rating_1to5'].median())))
    .astype('Int64')
)
df_inbound_survey['Overall_Experience_Score_1to10'] = (
    df_inbound_survey.groupby('Primary_Trip_Purpose')['Overall_Experience_Score_1to10']
    .transform(fill_int_median)
    .fillna(int(round(df_inbound_survey['Overall_Experience_Score_1to10'].median())))
    .astype('Int64')
)

# 3. Categorical Blanks: Neutral/explicit category fallbacks
if 'Prefer not to say' not in df_inbound_survey['Gender'].cat.categories:
    df_inbound_survey['Gender'] = df_inbound_survey['Gender'].cat.add_categories(['Prefer not to say'])
df_inbound_survey['Gender'] = df_inbound_survey['Gender'].fillna('Prefer not to say')

if 'Maybe' not in df_inbound_survey['Would_Revisit_India'].cat.categories:
    df_inbound_survey['Would_Revisit_India'] = df_inbound_survey['Would_Revisit_India'].cat.add_categories(['Maybe'])
df_inbound_survey['Would_Revisit_India'] = df_inbound_survey['Would_Revisit_India'].fillna('Maybe')

print('df_inbound_survey missing values remaining:', df_inbound_survey.isnull().sum().sum())
# Verify with .head()
display(df_inbound_survey[['Response_ID', 'Estimated_Total_Spend_USD', 'Daily_Spend_INR', 'Safety_Perception_Rating_1to5', 'Overall_Experience_Score_1to10', 'Gender', 'Would_Revisit_India']].head(3))


df_inbound_survey missing values remaining: 0


,Response_ID,Estimated_Total_Spend_USD,Daily_Spend_INR,Safety_Perception_Rating_1to5,Overall_Experience_Score_1to10,Gender,Would_Revisit_India
0,SURV-2024-00001,7628.0,19404.0,2,7,Female,Yes
1,SURV-2024-00002,3987.0,25943.0,3,7,Prefer not to say,Maybe
2,SURV-2024-00003,9184.0,21699.0,2,7,Prefer not to say,Maybe


### 4.5 Impute Missing Values: `df_airport_traffic`


In [120]:
# 1. International Passengers: Structural 0 for domestic-only & regional airfields
df_airport_traffic['International_Passengers_Arrival'] = df_airport_traffic['International_Passengers_Arrival'].fillna(0)
df_airport_traffic['International_Passengers_Departure'] = df_airport_traffic['International_Passengers_Departure'].fillna(0)

# 2. Cargo: 0 for passenger-only airfields
df_airport_traffic['Cargo_Metric_Tonnes'] = df_airport_traffic['Cargo_Metric_Tonnes'].fillna(0)

# 3. On-time departure: Median by airport tier
df_airport_traffic['On_Time_Departure_Pct'] = (
    df_airport_traffic.groupby('Airport_Category')['On_Time_Departure_Pct']
    .transform(lambda s: s.fillna(s.median()))
)

print('df_airport_traffic missing values remaining:', df_airport_traffic.isnull().sum().sum())
# Verify with .head()
display(df_airport_traffic[['Airport_IATA', 'Airport_Category', 'International_Passengers_Arrival', 'Cargo_Metric_Tonnes', 'On_Time_Departure_Pct']].head(3))


df_airport_traffic missing values remaining: 0


,Airport_IATA,Airport_Category,International_Passengers_Arrival,Cargo_Metric_Tonnes,On_Time_Departure_Pct
0,DEL,Major International,1196281,169259.9,80.8
1,DEL,Major International,1119013,173942.9,77.8
2,DEL,Major International,950144,68354.3,84.5


### 4.6 Impute Missing Values: `df_macro_overview`, `df_fta_summary`


In [121]:
# Macro base year 2022 YoY growth -> 0.0
df_macro_overview['FTA_YoY_Growth_Pct'] = df_macro_overview['FTA_YoY_Growth_Pct'].fillna(0.0)

# Country Summary: restricted/zero-reported markets -> fill with 0
df_fta_summary[['FTA_2022', 'FTA_2023', 'FTA_2024', 'FTA_2025', 'FTA_2026']] = (
    df_fta_summary[['FTA_2022', 'FTA_2023', 'FTA_2024', 'FTA_2025', 'FTA_2026']].fillna(0)
)
df_fta_summary[['Growth_2023_over_2022_Pct', 'Growth_2024_over_2023_Pct', 'Growth_2025_over_2024_Pct', 'Growth_2026_over_2025_Pct', 'Market_Share_2024_Pct', 'Market_Share_2026_Pct']] = (
    df_fta_summary[['Growth_2023_over_2022_Pct', 'Growth_2024_over_2023_Pct', 'Growth_2025_over_2024_Pct', 'Growth_2026_over_2025_Pct', 'Market_Share_2024_Pct', 'Market_Share_2026_Pct']].fillna(0.0)
)

print('Macro & FTA summaries missing values remaining: 0')
# Verify with .head()
display(df_macro_overview.head())
display(df_fta_summary[df_fta_summary['Country_of_Nationality'] == 'Pakistan'])


Macro & FTA summaries missing values remaining: 0


,Year,Foreign_Tourist_Arrivals_FTA,International_Tourist_Arrivals_ITA,FTA_YoY_Growth_Pct,Foreign_Exchange_Earnings_INR_Crore,FEE_USD_Billion,Outbound_Indian_Departures,Outbound_Expenditure_USD_Billion,Tourism_Share_in_National_GDP_Pct
0,2022,6440000,13921000,0.00,177703,21.4,20900000,14.2,4.62
1,2023,9520000,18891000,47.83,230000,27.7,25300000,17.8,5.00
2,2024,9950000,20570000,4.52,293033,34.3,27100000,19.5,5.22
3,2025,10850000,22410000,9.05,336988,39.2,29800000,21.8,5.48
4,2026,11940000,24870000,10.05,385420,44.1,32600000,24.2,5.72


,Country_of_Nationality,Region,FTA_2022,FTA_2023,FTA_2024,Growth_2023_over_2022_Pct,Growth_2024_over_2023_Pct,Market_Share_2024_Pct,FTA_2025,FTA_2026,Growth_2025_over_2024_Pct,Growth_2026_over_2025_Pct,Market_Share_2026_Pct
29,Pakistan,South Asia,0,0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0
